# 🚗 TakarMobil: Data Preprocessing & Pipeline Preparation (Phase 3)
*TakarMobil: Sistem Estimasi Rentang Harga & Analisis Faktor Penentu Harga Mobil Bekas*

---

### 🎯 Tujuan & Perbaikan Menyeluruh Tahap Preprocessing (Phase 3):
1. **Data Cleaning & Parsing:** Menghapus data tanpa harga, menyaring rentang harga wajar (Rp 25 Jt–10 M), menyaring 5 merek utama, parsing tahun (1995–2025), serta parsing kilometer (termasuk format `'K'` ribuan Mobil123). Menyatukan 1 iklan `'Listrik'` (*Mercedes EQS 450+*) ke `'Hybrid'` agar tidak menjadi kategori tunggal terisolasi.
2. **Aturan Wilayah Deterministik:** Memprioritaskan nama kota Jabodetabek eksplisit (`Jakarta`, `Bogor`, `Depok`, `Tangerang`, `Bekasi`) di dalam string `Lokasi` **sebelum** pencocokan nama kawasan/kecamatan `'% off'` (mengatasi 232 konflik seperti `"Jakarta Barat, Tangerang"` $\rightarrow$ `Jakarta` dan `"Jakarta Timur, Cibubur"` $\rightarrow$ `Jakarta`, sekaligus tetap memetakan `"Pondok Indah, 3% off"` $\rightarrow$ `Jakarta`).
3. **Deduplikasi & Kurasi Anomali Input:** Menghapus duplikat `URL` dan iklan repost identik `(Judul, Merek, Tahun, Harga, Lokasi)`, lalu mengeliminasi baris salah input yang terinspeksi:
   - Merek bukan target / salah label merek (*Smart for two*, *Lexus*, dan judul silang-merek).
   - Selisih tahun di judul vs `Tahun_Num` $\ge 5$ tahun (iklan DP / tahun salah input, setelah mengecualikan angka pajak & CC).
   - Unit tahun muda ($\ge 2023$) dengan odometer tak wajar ($> 80.000$ KM).
   - Salah ketik digit harga ekstrem ($< 0{,}25\times$ atau $> 4{,}0\times$ median `Merek + Model + Tahun`, mengecualikan varian performa/mewah asli seperti *GR Yaris*, *Civic Type R*, dan *Maybach*).
4. **Normalisasi Model Lintas Platform, Keluarga Mercedes & `Varian_Clean`:** Menyatukan penamaan model terpecah (*Brio Satya* $\rightarrow$ *Brio*, *Corolla/Altis* $\rightarrow$ *Corolla Altis*, *Vitara* $\rightarrow$ *Grand Vitara*, spasi, pemulihan *Fronx*/*SX4*), menurunkan 126 kode Mercedes ke keluarga model (`C-Class`, `E-Class`, `S-Class`, `GLC-Class`, dst.), dan mengekstrak `Varian_Clean` untuk detail trim/kode (`C200`, `C300`, `AMG Line`, `RS`, `G`, `VRZ`).
5. **Rekayasa Fitur Bebas Redundansi:** Membentuk **`KM_hilang`** (`0`/`1` untuk menangkap sinyal kekosongan KM non-acak di OLX), mempertahankan **satu** fitur waktu (**`Usia_Mobil = 2025 - Tahun_Num`** tanpa kolinearitas sempurna `Tahun_Num`), dan target **`Log_Harga`** ($\log(1 + \text{Harga})$).
6. **Splitting Anti-Leakage Ganda (`StratifiedGroupKFold` Berbasis Union-Find):** Mengelompokkan iklan menggunakan **Union-Find** atas **kunci tanpa harga** `(Merek, Model, Tahun, Varian, Warna, Lokasi, Transmisi, KM)` sekaligus **kunci dengan harga** `(Merek, Model, Tahun, Harga)` agar **nol iklan kembar bocor** di kedua definisi antara Train (80%) dan Test (20%).
7. **Imputasi KM Pasca-Split Khusus Data Training:** Menghitung tabel median bertingkat (`Model + Tahun` $\rightarrow$ `Merek + Tahun` $\rightarrow$ Global) **eksklusif dari `train_df`** lalu menerapkannya ke `train_df` & `test_df` (*0 test leakage*). Berkas `clean_cars_data.csv` sengaja **tidak** menyertakan kolom `KM_Clean` hasil statistik train.

## 1. Import Library & Setup Lingkungan

In [1]:
import os
import re
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')
print("Pustaka preprocessing berhasil dimuat!")

Pustaka preprocessing berhasil dimuat!


## 2. Pemuatan Dataset Mentah (Raw Data)

In [2]:
candidate_paths = [
    "data/raw/Data Scraping.xlsx",
    "../data/raw/Data Scraping.xlsx",
    "Data Scraping.xlsx"
]
file_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if file_path is None:
    raise FileNotFoundError("File Data Scraping.xlsx tidak ditemukan!")

print(f"Memuat dataset mentah dari: {file_path}")
df_raw = pd.read_excel(file_path)
print(f"Jumlah baris awal: {df_raw.shape[0]}, Jumlah kolom: {df_raw.shape[1]}")
df_raw.head(3)

Memuat dataset mentah dari: ../data/raw/Data Scraping.xlsx


Jumlah baris awal: 17670, Jumlah kolom: 16


,Sumber,URL,Judul,Merek,Model,Varian,Tahun,Kilometer,Transmisi,Warna,Kapasitas Mesin,Bahan Bakar,Tanggal Upload,Lokasi,Tipe Bodi,Harga
0,OLX,https://www.olx.co.id/item/939502279,MERCEDES BENZ A45 AMG 2014 2.0 TURBO AWD PANOR...,Mercedes-Benz,A45 AMG,2.0 Bensin-AT,2014,NaN,Automatic,Putih,NaN,Bensin,07-10-2025,Jakarta Timur,NaN,499.000.000
1,OLX,https://www.olx.co.id/item/939871468,Mercedes Benz Sprinter A415 CDI 2022,Mercedes-Benz,Sprinter,NaN,2021,NaN,Manual,Hitam,NaN,Diesel,17-10-2025,Jakarta Utara,NaN,1.850.000.000
2,OLX,https://www.olx.co.id/item/939450820,[ GARANSI 5TH ] Mercedes Benz Mercy GLC200 GLC...,Mercedes-Benz,GLC200,2.0 AMG Line Bensin-AT,2022,NaN,Automatic,Putih,NaN,Bensin,06-10-2025,Jakarta Barat,NaN,720.000.000


## 3. Data Cleaning Dasar, Wilayah Deterministik & Deduplikasi Awal
Menangani parsing numerik, filter harga (Rp 25 Jt–10 M), filter 5 merek utama, tahun (1995–2025), pemetaan `Wilayah` yang memprioritaskan nama kota eksplisit sebelum kata kunci kawasan `'% off'`, serta penghapusan duplikat `URL` dan iklan repost identik.

In [3]:
df = df_raw.copy()

# A. Parsing & Filter Harga
def clean_price(val):
    if pd.isna(val): return np.nan
    if isinstance(val, (int, float)):
        v = float(val)
    else:
        digits = ''.join(c for c in str(val).replace('.', '').strip() if c.isdigit())
        v = float(digits) if digits else np.nan
    return v if (not pd.isna(v) and v >= 10_000_000) else np.nan

df['Harga_Num'] = df['Harga'].apply(clean_price)
df = df.dropna(subset=['Harga_Num'])
df = df[(df['Harga_Num'] >= 25_000_000) & (df['Harga_Num'] <= 10_000_000_000)]

# B. Standarisasi 5 Merek Utama
mercedes_series = [
    'C-Class', 'E-Class', 'GLC-Class', 'GLA-Class', 'CLA-Class', 'GLE-Class',
    'A-Class', 'S-Class', 'GLS-Class', 'V-Class', 'CLS-Class', 'SLK-Class',
    'GLB-Class', 'B-Class', 'ML-Class', 'G-Class', 'GL-Class', 'CLS400',
    'CLA200', 'SL-Class', 'Mercedes Benz'
]
def standardize_brand(m):
    if pd.isna(m): return 'Lainnya'
    ms = str(m).strip()
    if ms in mercedes_series or 'mercedes' in ms.lower() or 'benz' in ms.lower(): return 'Mercedes-Benz'
    for b in ['Toyota', 'Daihatsu', 'Suzuki', 'Honda']:
        if b.lower() in ms.lower(): return b
    return ms

df['Merek_Clean'] = df['Merek'].apply(standardize_brand)
TARGET_BRANDS = ['Toyota', 'Daihatsu', 'Suzuki', 'Honda', 'Mercedes-Benz']
df = df[df['Merek_Clean'].isin(TARGET_BRANDS)]

# C. Standarisasi Transmisi & Bahan Bakar (1 unit 'Listrik' digabung ke 'Hybrid')
def clean_transmission(val):
    if pd.isna(val): return 'Automatic'
    v = str(val).lower()
    if 'man' in v or 'mt' in v: return 'Manual'
    return 'Automatic'
df['Transmisi_Clean'] = df['Transmisi'].apply(clean_transmission)

listrik_count = int((df['Bahan Bakar'].astype(str).str.lower() == 'listrik').sum())
def clean_fuel(val):
    if pd.isna(val): return 'Bensin'
    v = str(val).lower()
    if 'diesel' in v or 'solar' in v: return 'Diesel'
    if 'hybrid' in v or 'listrik' in v or 'electric' in v: return 'Hybrid'
    return 'Bensin'
df['Bahan_Bakar_Clean'] = df['Bahan Bakar'].apply(clean_fuel)
print(f"Catatan Bahan Bakar: {listrik_count} unit berlabel 'Listrik' disatukan ke kategori 'Hybrid'.")

# D. Parsing Kilometer (menangani '10 - 15K km' Mobil123 & rentang OLX)
def clean_km(val):
    if pd.isna(val): return np.nan
    s = str(val).lower().replace('km', '').strip()
    is_k = 'k' in s
    s = s.replace('k', '').strip()
    nums = []
    for p in s.split('-'):
        d = ''.join(c for c in p.replace('.', '').replace(',', '').strip() if c.isdigit())
        if d:
            v = float(d)
            if is_k or v <= 500:
                v *= 1000.0
            nums.append(v)
    if not nums: return np.nan
    res = float(np.mean(nums))
    return res if 0 <= res <= 400_000 else np.nan
df['KM_Num'] = df['Kilometer'].apply(clean_km)

# E. Parsing Tahun
def clean_year(val):
    if pd.isna(val): return np.nan
    d = ''.join(c for c in str(val) if c.isdigit())
    if d and len(d) == 4:
        y = int(d)
        if 1995 <= y <= 2025: return y
    return np.nan
df['Tahun_Num'] = df['Tahun'].apply(clean_year)
df = df.dropna(subset=['Tahun_Num'])
df['Tahun_Num'] = df['Tahun_Num'].astype(int)

# F. Aturan Wilayah Deterministik:
#    1) Cek kemunculan NAMA KOTA pertama secara eksplisit (Jakarta, Bogor, Depok, Tangerang, Bekasi)
#       sehingga 'Jakarta Barat, Tangerang' -> Jakarta dan 'Jakarta Timur, Cibubur' -> Jakarta.
#    2) Jika tidak ada nama kota eksplisit (format 'Pondok Indah, 3% off' Mobil123), pakai kamus kawasan.
#    3) Sisanya -> 'Luar Jabodetabek'.
CITY_REGEX = re.compile(r'\b(jakarta|bogor|depok|tangerang|bekasi)\b')
TANGERANG_KEYWORDS = [
    'tangerang', 'alam sutera', 'pamulang', 'banjar wijaya', 'batu ceper',
    'bsd', 'cikupa', 'ciledug', 'cipondoh', 'ciputat', 'citra raya',
    'gading serpong', 'jatake', 'jatiuwung', 'jurangmangu', 'karawaci',
    'pondok cabe', 'serpong'
]
BEKASI_KEYWORDS = [
    'bekasi', 'bintara', 'cibitung', 'harapan indah', 'jatiwaringin',
    'jatiwarna', 'kartini', 'medan satria', 'pondok gede'
]
DEPOK_KEYWORDS = ['depok', 'cimanggis', 'kelapa dua']
BOGOR_KEYWORDS = ['bogor', 'cibinong', 'cibubur', 'cileungsi']
JAKARTA_KEYWORDS = [
    'jakarta', 'ancol', 'batu tulis', 'bintaro', 'buaran', 'cakung',
    'cempaka putih', 'cengkareng', 'cijantung', 'cilandak', 'cilincing',
    'cipinang', 'ciracas', 'daan mogot', 'dewi sartika', 'duren sawit',
    'duri kosambi', 'fatmawati', 'grogol', 'gunung sahari', 'jalan panjang',
    'kalideres', 'kebayoran', 'kebon jeruk', 'kelapa gading', 'kemang',
    'kemayoran', 'kramat jati', 'kuningan', 'lebak bulus', 'lubang buaya',
    'mangga besar', 'mangga dua', 'pademangan', 'pancoran',
    'pantai indah kapuk', 'pasar minggu', 'pasar rebo', 'permata hijau',
    'pluit', 'pondok indah', 'pulo gebang', 'puri indah', 'sawah besar',
    'setiabudi', 'sunter', 'tamansari', 'taman sari', 'tambora'
]

def simplify_location(loc):
    if pd.isna(loc): return 'Jakarta'
    l = str(loc).lower()
    m_city = CITY_REGEX.search(l)
    if m_city:
        return m_city.group(1).title()
    if any(k in l for k in TANGERANG_KEYWORDS): return 'Tangerang'
    if any(k in l for k in BEKASI_KEYWORDS): return 'Bekasi'
    if any(k in l for k in DEPOK_KEYWORDS): return 'Depok'
    if any(k in l for k in BOGOR_KEYWORDS): return 'Bogor'
    if any(k in l for k in JAKARTA_KEYWORDS): return 'Jakarta'
    return 'Luar Jabodetabek'

df['Wilayah'] = df['Lokasi'].apply(simplify_location)

# G. Deduplikasi URL & Iklan Repost Identik
before_dedup = len(df)
df = df.drop_duplicates(subset=['URL']).reset_index(drop=True)
after_url = len(df)
df = df.drop_duplicates(subset=['Judul', 'Merek_Clean', 'Tahun_Num', 'Harga_Num', 'Lokasi']).reset_index(drop=True)

print(f"Sebelum deduplikasi : {before_dedup} baris")
print(f"Setelah hapus URL   : {after_url} baris (-{before_dedup - after_url})")
print(f"Setelah hapus repost: {len(df)} baris (-{after_url - len(df)})")
print("Distribusi Wilayah:", df['Wilayah'].value_counts().to_dict())

Catatan Bahan Bakar: 1 unit berlabel 'Listrik' disatukan ke kategori 'Hybrid'.


Sebelum deduplikasi : 16799 baris
Setelah hapus URL   : 14754 baris (-2045)
Setelah hapus repost: 13359 baris (-1395)
Distribusi Wilayah: {'Jakarta': 9085, 'Tangerang': 1393, 'Bekasi': 975, 'Bogor': 753, 'Luar Jabodetabek': 682, 'Depok': 471}


## 4. Kurasi & Penghapusan Baris Salah Input (Inspeksi Manual)
Sesuai temuan audit data, dilakukan pemeriksaan eksplisit terhadap empat sumber kesalahan entri iklan yang terbukti keliru:
1. **Salah Merek / Non-Target:** *Smart for two 2010* (masuk ke Mercedes-Benz), *Lexus LX570* (masuk ke Toyota), serta iklan yang judulnya menyebut merek lain tanpa menyebut merek berlabel.
2. **Salah Tahun Produksi ($\ge 5$ tahun selisih terhadap judul):** Iklan kredit/DP yang memasang tahun salah di kolom `Tahun` (mis. *Camry 2002* tertulis `2020`, *Jazz 2004* tertulis `2018`, *Vellfire 2009* tertulis `2019`), mengecualikan angka pajak (`Tax/Pajak/Pjk 2026`) & kapasitas mesin (`2000 cc`).
3. **Odometer Tak Wajar pada Mobil Baru:** Baris `Tahun_Num >= 2023` dengan `KM_Num > 80.000` (mis. *Sigra 2024 227.500 km*).
4. **Salah Ketik Digit Harga Ekstrem:** Harga $< 0{,}25\times$ atau $> 4{,}0\times$ median `Merek + Model mentah + Tahun` (mis. *Yaris 2019 Rp 1,99 M*, *Karimun Wagon R 2020 Rp 880 jt*, *S450L 2022 Rp 385 jt*, *E250 2012 Rp 4,247 M*, *C250 2015 Rp 37,9 jt*), **mengecualikan** varian edisi khusus/Maybach/GR Yaris/Type R serta unit *S400 2015* valid.

In [4]:
def is_wrong_brand(r):
    j = str(r['Judul']).lower()
    mr = str(r['Model']).lower()
    if 'smart' in j or 'lexus' in j or 'lexus' in mr:
        return True
    mb = r['Merek_Clean']
    mbt = ['mercedes', 'mercy'] if mb == 'Mercedes-Benz' else [mb.lower()]
    if any(re.search(r'\b' + t + r'\b', j) for t in mbt):
        return False
    other_brands = ['toyota', 'daihatsu', 'suzuki', 'honda', 'bmw', 'audi', 'mitsubishi', 'nissan', 'wuling', 'hyundai', 'mazda']
    return any(ob != mb.lower() and re.search(r'\b' + ob + r'\b', j) for ob in other_brands)

def extract_vehicle_years(j):
    s = re.sub(r'(?:pajak|tax|pjk)\s*[:=\-]?\s*(?:[a-z]+\s*[-\s]*)?(?:\d{1,2}\s*[-\/]\s*)?\d{4}', ' ', str(j).lower())
    s = re.sub(r'\b\d{4}\s*cc\b', ' ', s)
    return [int(y) for y in re.findall(r'\b(199\d|20[0-2]\d)\b', s)]

m_brand = df.apply(is_wrong_brand, axis=1)
m_year = df.apply(lambda r: bool(extract_vehicle_years(r['Judul'])) and min(abs(r['Tahun_Num'] - y) for y in extract_vehicle_years(r['Judul'])) >= 5, axis=1)
m_km_new = (df['Tahun_Num'] >= 2023) & (df['KM_Num'] > 80_000)

# Cek salah ketik harga ekstrem berbasis model mentah (kecualikan S400 valid di S-Class, Maybach, GR Yaris, Type R)
med_raw = df.groupby(['Merek_Clean', 'Model', 'Tahun_Num'])['Harga_Num'].transform('median')
ratio_raw = df['Harga_Num'] / med_raw
is_legit_special = (
    df['Judul'].astype(str).str.contains(r'(?:gr yaris|type r|maybach|s400)', case=False, na=False) |
    df['Varian'].astype(str).str.contains(r'maybach', case=False, na=False)
)
m_price_typo = ((ratio_raw > 4.0) | (ratio_raw < 0.25)) & (~is_legit_special)

drop_mask = m_brand | m_year | m_km_new | m_price_typo
print(f"Ringkasan Penghapusan Anomali (A6) -> Salah Merek: {m_brand.sum()} | Salah Tahun Judul: {m_year.sum()} | KM>80k Th>=2023: {m_km_new.sum()} | Salah Ketik Harga: {m_price_typo.sum()}")
print(f"Total baris anomali dibuang: {drop_mask.sum()} -> Sisa data terkurasi: {(~drop_mask).sum()} baris")

df = df[~drop_mask].copy().reset_index(drop=True)

Ringkasan Penghapusan Anomali (A6) -> Salah Merek: 15 | Salah Tahun Judul: 13 | KM>80k Th>=2023: 9 | Salah Ketik Harga: 7
Total baris anomali dibuang: 39 -> Sisa data terkurasi: 13320 baris


## 5. Normalisasi Model Lintas Platform, Keluarga Mercedes, `Varian_Clean` & Rekayasa Fitur
- **Normalisasi `Model_Clean`:** Menggabungkan sinonim & variasi spasi (*Brio Satya* $\rightarrow$ *Brio*, *Altis/Corolla* $\rightarrow$ *Corolla Altis*, *Vitara* $\rightarrow$ *Grand Vitara*, pemulihan *Lain-lain* menjadi *Fronx*/*SX4* via judul), dan memetakan seluruh 126 kode Mercedes ke keluarga model (`C-Class`, `E-Class`, `S-Class`, `GLC-Class`, dst.).
- **Ekstraksi `Varian_Clean`:** Membersihkan akhiran platform (`Bensin-AT`, `Diesel-MT`, kapasitas awal) sekaligus menyematkan kode seri Mercedes (`C200`, `C300`, `E300`, `G63`, `S450L`, dll.) agar detail trim tetap terjaga.
- **Fitur Bebas Redundansi:** Membentuk **`KM_hilang`** (`0`/`1`), target **`Log_Harga`**, dan memilih satu fitur umur kendaraan (**`Usia_Mobil = 2025 - Tahun_Num`**) guna menghilangkan multikolinearitas sempurna dengan `Tahun_Num`.

In [5]:
MODEL_SYNONYMS = {
    ('Honda', 'Brio Satya'): 'Brio',
    ('Honda', 'Civic Hatchback'): 'Civic',
    ('Toyota', 'Altis'): 'Corolla Altis',
    ('Toyota', 'Corolla'): 'Corolla Altis',
    ('Toyota', 'Etios'): 'Etios Valco',
    ('Toyota', 'Ft'): 'FT86',
    ('Toyota', 'Ft86'): 'FT86',
    ('Toyota', 'Prado'): 'Land Cruiser Prado',
    ('Daihatsu', 'Grand Xenia'): 'Xenia',
    ('Daihatsu', 'Gran Max Mb'): 'Gran Max',
    ('Daihatsu', 'Gran Max Pick-Up'): 'Gran Max',
    ('Daihatsu', 'Gran Max Pu'): 'Gran Max',
    ('Suzuki', 'Vitara'): 'Grand Vitara',
    ('Suzuki', 'Apv Arena'): 'APV',
    ('Suzuki', 'Arena'): 'APV',
    ('Suzuki', 'Apv'): 'APV',
    ('Suzuki', 'Carry Pick-Up'): 'Carry',
    ('Suzuki', 'Xl7'): 'XL7',
    ('Suzuki', 'Sx4'): 'SX4',
    ('Honda', 'Cr-V'): 'CR-V',
    ('Honda', 'Hr-V'): 'HR-V',
    ('Honda', 'Br-V'): 'BR-V',
    ('Honda', 'Wr-V'): 'WR-V',
    ('Toyota', 'C-Hr'): 'C-HR',
}

MERCEDES_FAMILY_PREFIXES = [
    ('glc', 'GLC-Class'), ('gle', 'GLE-Class'), ('gla', 'GLA-Class'), ('glb', 'GLB-Class'),
    ('gls', 'GLS-Class'), ('gl', 'GL-Class'), ('cla', 'CLA-Class'), ('cls', 'CLS-Class'),
    ('clc', 'CLC-Class'), ('slk', 'SLK-Class'), ('slc', 'SLC-Class'), ('sl', 'SL-Class'),
    ('ml', 'ML-Class'), ('eqs', 'EQS'), ('gt', 'AMG GT'), ('sprinter', 'Sprinter'),
    ('viano', 'Viano'), ('vito', 'Vito'), ('c', 'C-Class'), ('e', 'E-Class'),
    ('s', 'S-Class'), ('g', 'G-Class'), ('a', 'A-Class'), ('b', 'B-Class'),
    ('v', 'V-Class'), ('r', 'R-Class')
]

def normalize_model(row):
    b = row['Merek_Clean']
    m = (str(row['Model']).strip() if pd.notna(row['Model']) else 'Lainnya').title()
    j = str(row['Judul']).lower() if pd.notna(row['Judul']) else ''
    if m.lower() in ('lain-lain', 'lainnya'):
        if 'fronx' in j or 'fronk' in j: m = 'Fronx'
        elif 'croos' in j or 'sx4' in j or 's-cross' in j: m = 'SX4'
        elif 'c200' in j or 'c300' in j or 'c250' in j: m = 'C-Class'
    if (b, m) in MODEL_SYNONYMS:
        return MODEL_SYNONYMS[(b, m)]
    if b == 'Mercedes-Benz':
        s = re.sub(r'^amg\s+', '', m.lower().strip()).replace(' ', '')
        for pfx, fam in MERCEDES_FAMILY_PREFIXES:
            if s.startswith(pfx):
                return fam
    return m

def clean_varian_norm(row):
    b = row['Merek_Clean']
    rm = str(row['Model']).strip() if pd.notna(row['Model']) else ''
    v = str(row['Varian']).strip() if pd.notna(row['Varian']) else ''
    v = re.sub(r'\b(Bensin|Diesel|Solar|Hybrid|Listrik)[-\s]*(AT|MT|A/T|M/T)\b', '', v, flags=re.I)
    v = re.sub(r'\b(Bensin|Diesel|Solar|Hybrid|Listrik|AT|MT|Lain-lain)\b', '', v, flags=re.I)
    v = re.sub(r'^\d\.\d\s*', '', v).strip()
    v = re.sub(r'\s+', ' ', v).strip(' -/')
    if b == 'Mercedes-Benz':
        mc = re.search(r'\b([A-Za-z]{1,3}\s?\d{2,3}[A-Za-z]?)\b', rm.replace('-', ' '))
        code = mc.group(1).replace(' ', '').upper() if mc else ''
        if code and code not in v.upper().replace(' ', ''):
            v = f"{code} {v}".strip() if v else code
        elif v:
            v = re.sub(r'\b([A-Za-z]{1,3})\s+(\d{2,3}[A-Za-z]?)\b', r'\1\2', v)
    return v.title() if v else 'Standar'

df['Model_Clean'] = df.apply(normalize_model, axis=1)
df['Varian_Clean'] = df.apply(clean_varian_norm, axis=1)

# Fitur waktu tunggal (Usia_Mobil acuan 2025), indikator KM hilang, target Log_Harga
TAHUN_ACUAN = 2025
df['Usia_Mobil'] = TAHUN_ACUAN - df['Tahun_Num']
df['KM_hilang'] = df['KM_Num'].isna().astype(int)
df['Log_Harga'] = np.log1p(df['Harga_Num'])
df['Merek_Model'] = df['Merek_Clean'] + "_" + df['Model_Clean']

mm_src = df.groupby(['Merek_Clean', 'Model_Clean'])['Sumber'].nunique()
print(f"Jumlah Model_Clean unik setelah normalisasi: {df['Model_Clean'].nunique()} (tinggal {(mm_src==1).sum()} yang single-platform)")
print(f"Jumlah Varian_Clean unik sebelum threshold OHE: {df['Varian_Clean'].nunique()}")
df[['Merek_Clean', 'Model_Clean', 'Varian_Clean', 'Tahun_Num', 'Usia_Mobil', 'KM_Num', 'KM_hilang', 'Wilayah', 'Harga_Num']].head()

Jumlah Model_Clean unik setelah normalisasi: 105 (tinggal 24 yang single-platform)
Jumlah Varian_Clean unik sebelum threshold OHE: 675


,Merek_Clean,Model_Clean,Varian_Clean,Tahun_Num,Usia_Mobil,KM_Num,KM_hilang,Wilayah,Harga_Num
0,Mercedes-Benz,A-Class,A45,2014,11,NaN,1,Jakarta,4.990000e+08
1,Mercedes-Benz,Sprinter,Standar,2021,4,NaN,1,Jakarta,1.850000e+09
2,Mercedes-Benz,GLC-Class,Glc200 Amg Line,2022,3,NaN,1,Jakarta,7.200000e+08
3,Mercedes-Benz,GLE-Class,Gle400 Coupe Amg Line,2016,9,NaN,1,Jakarta,5.980000e+08
4,Mercedes-Benz,ML-Class,Ml250 Cdi,2016,9,NaN,1,Tangerang,3.560000e+08


## 6. Pembagian Data Anti-Leakage Ganda: Union-Find + `StratifiedGroupKFold` (80:20)
Menghilangkan celah kebocoran di kedua definisi sekaligus dengan membentuk komponen terhubung (**Union-Find**) dari:
1. **Kunci Identitas Fisik Tanpa Harga:** `(Merek_Clean, Model_Clean, Tahun_Num, Varian, Warna, Lokasi, Transmisi_Clean, KM_Num)`.
2. **Kunci Iklan Kembar Dengan Harga:** `(Merek_Clean, Model_Clean, Tahun_Num, Harga_Num)`.

Kemudian dataset dibagi 80:20 menggunakan **`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`** dengan `groups = dup_group` (dilakukan **sebelum** imputasi KM).

In [6]:
n = len(df)
parent = list(range(n))
def _find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb:
        parent[rb] = ra

k_noprice = list(zip(
    df['Merek_Clean'], df['Model_Clean'], df['Tahun_Num'],
    df['Varian'].fillna('__NA__').astype(str), df['Warna'].fillna('__NA__').astype(str),
    df['Lokasi'].fillna('__NA__').astype(str), df['Transmisi_Clean'], df['KM_Num'].fillna(-1)
))
k_withprice = list(zip(df['Merek_Clean'], df['Model_Clean'], df['Tahun_Num'], df['Harga_Num']))

seen_a, seen_b = {}, {}
for i in range(n):
    ka, kb = k_noprice[i], k_withprice[i]
    if ka in seen_a: _union(seen_a[ka], i)
    else: seen_a[ka] = i
    if kb in seen_b: _union(seen_b[kb], i)
    else: seen_b[kb] = i

df['dup_group'] = [f"G{_find(i)}" for i in range(n)]

grp_counts = df.groupby('Merek_Model')['dup_group'].nunique()
strat_key = df['Merek_Model'].where(df['Merek_Model'].map(grp_counts) >= 5, 'RARE_MODEL')

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, strat_key, groups=df['dup_group']))

train_df = df.iloc[train_idx].copy().reset_index(drop=True)
test_df = df.iloc[test_idx].copy().reset_index(drop=True)

def _key_set(part, cols):
    s = part.copy()
    for c in cols: s[c] = s[c].fillna('__NA__').astype(str)
    return set(map(tuple, s[cols].values))

ov_group = set(train_df['dup_group']) & set(test_df['dup_group'])
ov_noprice = _key_set(train_df, ['Merek_Clean','Model_Clean','Tahun_Num','Varian','Warna','Lokasi','Transmisi_Clean','KM_Num']) & \
             _key_set(test_df, ['Merek_Clean','Model_Clean','Tahun_Num','Varian','Warna','Lokasi','Transmisi_Clean','KM_Num'])
ov_withprice = _key_set(train_df, ['Merek_Clean','Model_Clean','Tahun_Num','Harga_Num']) & \
               _key_set(test_df, ['Merek_Clean','Model_Clean','Tahun_Num','Harga_Num'])

assert len(ov_group) == 0 and len(ov_noprice) == 0 and len(ov_withprice) == 0

print(f"Total Data Terkurasi : {len(df)} baris ({df['dup_group'].nunique()} grup unik Union-Find)")
print(f"Data Training        : {len(train_df)} baris ({len(train_df)/len(df)*100:.2f}%)")
print(f"Data Testing         : {len(test_df)} baris ({len(test_df)/len(df)*100:.2f}%)")
print(f"Verifikasi Leakage   -> dup_group overlap: {len(ov_group)} | kunci tanpa harga: {len(ov_noprice)} | kunci dengan harga: {len(ov_withprice)}")

Total Data Terkurasi : 13320 baris (6752 grup unik Union-Find)
Data Training        : 10656 baris (80.00%)
Data Testing         : 2664 baris (20.00%)
Verifikasi Leakage   -> dup_group overlap: 0 | kunci tanpa harga: 0 | kunci dengan harga: 0


## 7. Imputasi Missing Values KM (Pasca-Split, Khusus Statistik Data Training)
Tabel median kilometer bertingkat (`Model_Clean + Tahun_Num` $\\rightarrow$ `Merek_Clean + Tahun_Num` $\\rightarrow$ Global) dihitung **hanya dari `train_df`**. Kolom `KM_Clean` diisi hanya pada `train_data.csv` dan `test_data.csv` (*tidak ditulis ke `clean_cars_data.csv` agar aman bila `clean_cars_data.csv` diperiksa terpisah*).

In [7]:
print(f"Missing KM sebelum imputasi -> Train: {train_df['KM_Num'].isnull().sum()} | Test: {test_df['KM_Num'].isnull().sum()}")

km_med_model_year = train_df.groupby(['Model_Clean', 'Tahun_Num'])['KM_Num'].median().to_dict()
km_med_brand_year = train_df.groupby(['Merek_Clean', 'Tahun_Num'])['KM_Num'].median().to_dict()
km_med_global = float(train_df['KM_Num'].median())

def apply_train_km_imputation(part_df):
    my_fill = pd.Series(
        [km_med_model_year.get((m, t), np.nan) for m, t in zip(part_df['Model_Clean'], part_df['Tahun_Num'])],
        index=part_df.index, dtype='float64'
    )
    by_fill = pd.Series(
        [km_med_brand_year.get((b, t), np.nan) for b, t in zip(part_df['Merek_Clean'], part_df['Tahun_Num'])],
        index=part_df.index, dtype='float64'
    )
    return part_df['KM_Num'].fillna(my_fill).fillna(by_fill).fillna(km_med_global)

train_df['KM_Clean'] = apply_train_km_imputation(train_df)
test_df['KM_Clean'] = apply_train_km_imputation(test_df)

assert train_df['KM_Clean'].isnull().sum() == 0 and test_df['KM_Clean'].isnull().sum() == 0
print(f"Missing KM setelah imputasi  -> Train: {train_df['KM_Clean'].isnull().sum()} | Test: {test_df['KM_Clean'].isnull().sum()}")
print(f"Global median KM (dari train_df): {km_med_global:,.0f} KM (0 statistik test terpakai)")

Missing KM sebelum imputasi -> Train: 2345 | Test: 579
Missing KM setelah imputasi  -> Train: 0 | Test: 0
Global median KM (dari train_df): 52,500 KM (0 statistik test terpakai)


## 8. Pipeline Scikit-Learn (`ColumnTransformer`) & Ekspor Artefak
Mempersiapkan `ColumnTransformer` dengan fitur numerik (`Usia_Mobil`, `KM_Clean`, `KM_hilang`) dan kategorikal (`Merek_Clean`, `Model_Clean`, `Varian_Clean`, `Transmisi_Clean`, `Bahan_Bakar_Clean`, `Wilayah`) menggunakan `OneHotEncoder(handle_unknown='ignore', min_frequency=5)` yang di-fit **eksklusif pada `X_train`**, lalu mengekspor seluruh dataset dan `feature_metadata.json`.

In [8]:
num_features = ['Usia_Mobil', 'KM_Clean', 'KM_hilang']
cat_features = ['Merek_Clean', 'Model_Clean', 'Varian_Clean', 'Transmisi_Clean', 'Bahan_Bakar_Clean', 'Wilayah']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=5, sparse_output=False), cat_features)
    ],
    remainder='drop'
)

X_train = train_df[num_features + cat_features]
y_train = train_df['Log_Harga']
preprocessor.fit(X_train)
X_train_t = preprocessor.transform(X_train)
print(f"ColumnTransformer berhasil di-fit pada X_train: {X_train_t.shape[0]} baris x {X_train_t.shape[1]} kolom fitur ter-encode.")

proc_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
models_dir = '../models' if os.path.exists('../models') else 'models'
os.makedirs(proc_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

clean_path = os.path.join(proc_dir, 'clean_cars_data.csv')
train_path = os.path.join(proc_dir, 'train_data.csv')
test_path = os.path.join(proc_dir, 'test_data.csv')
df.to_csv(clean_path, index=False)
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)
joblib.dump(preprocessor, os.path.join(models_dir, 'preprocessor.joblib'))

models_by_brand = {}
for brand, grp in df.groupby('Merek_Clean'):
    vc_b = grp['Model_Clean'].value_counts()
    models_by_brand[brand] = [{'name': str(m), 'count': int(c)} for m, c in vc_b.items()]

variants_by_model = {}
for (brand, model), grp in df.groupby(['Merek_Clean', 'Model_Clean']):
    vc_v = grp['Varian_Clean'].value_counts()
    variants_by_model[f"{brand}||{model}"] = [{'name': str(v), 'count': int(c)} for v, c in vc_v.items()]

wilayah_counts = {str(k): int(v) for k, v in df['Wilayah'].value_counts().items()}

metadata = {
    'num_features': num_features,
    'cat_features': cat_features,
    'target_column': 'Harga_Num',
    'log_target_column': 'Log_Harga',
    'tahun_acuan': TAHUN_ACUAN,
    'brands': TARGET_BRANDS,
    'models_by_brand': models_by_brand,
    'variants_by_model': variants_by_model,
    'wilayah_list': sorted(df['Wilayah'].unique().tolist()),
    'wilayah_counts': wilayah_counts,
    'transmisi_list': sorted(df['Transmisi_Clean'].unique().tolist()),
    'bahan_bakar_list': sorted(df['Bahan_Bakar_Clean'].unique().tolist()),
    'total_clean_samples': int(len(df)),
    'train_samples': int(len(train_df)),
    'test_samples': int(len(test_df)),
    'unique_groups_union_find': int(df['dup_group'].nunique()),
    'overlap_with_price': int(len(ov_withprice)),
    'overlap_no_price': int(len(ov_noprice))
}

with open(os.path.join(models_dir, 'feature_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print(f"Ekspor selesai -> Total bersih: {len(df)} | Train: {len(train_df)} | Test: {len(test_df)}")

ColumnTransformer berhasil di-fit pada X_train: 10656 baris x 368 kolom fitur ter-encode.


Ekspor selesai -> Total bersih: 13320 | Train: 10656 | Test: 2664


## 9. Ringkasan Audit & Checklist Preprocessing (Phase 3)

| Komponen Preprocessing | Status | Implementasi Konkret |
|---|---|---|
| **Pembersihan & Wilayah** | ✅ Selesai | Filter 5 merek, Rp 25 Jt–10 M, 1995–2025, `KM` `'K'`, catat 1 unit `'Listrik'` $\rightarrow$ `'Hybrid'`, serta prioritaskan nama kota eksplisit sebelum kata kunci kawasan `'% off'`. |
| **Deduplikasi & Anomali A6** | ✅ Selesai | Hapus duplikat `URL` + repost identik + kurasi baris salah merek (`Smart`/`Lexus`/judul silang-merek), salah tahun judul $\ge 5$ th, `KM>80k` untuk $\ge 2023$, dan salah ketik digit harga ekstrem. |
| **Normalisasi Model & Trim** | ✅ Selesai | Satukan sinonim lintas-platform (`Brio Satya`, `Corolla Altis`, `Grand Vitara`, spasi), turunkan keluarga Mercedes, dan ekstrak `Varian_Clean`. |
| **Fitur Bebas Redundansi** | ✅ Selesai | Tambahkan **`KM_hilang`** (`0`/`1`), gunakan **`Usia_Mobil` (`2025 - Tahun_Num`)** saja (*0 kolinearitas sempurna*), dan target `Log_Harga`. |
| **Splitting Anti-Leakage Ganda** | ✅ Selesai | Union-Find atas **kunci tanpa harga** & **kunci dengan harga** + **`StratifiedGroupKFold` (80:20)** $\rightarrow$ `0` overlap grup di kedua definisi. |
| **Imputasi Pasca-Split** | ✅ Selesai | Median KM bertingkat dihitung **hanya dari `train_df`**; `clean_cars_data.csv` dibiarkan tanpa kolom `KM_Clean` agar tidak membocorkan statistik train bila di-split ulang. |